In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# This interactive notebook visualizes ALL 2N poles of the function
#
#       H(s)H(-s)
#
# associated with a continuous-time Chebyshev Type II low-pass filter.
#
# Three parameters can be varied:
#
#       N     : filter order
#       ωs    : stopband-edge angular frequency
#       As    : minimum stopband attenuation in dB
#
# The ripple parameter is
#
#       ε = 1 / sqrt(10^(As/10) - 1)
#
# The poles do NOT generally lie on a circle, ellipse or hyperbola.
# Their geometric locus is more complicated.
#
# The poles are related to the corresponding Chebyshev Type I poles
# through an inversion-type transformation.
#
# Pole representation:
#
#       Filled red circles : poles used in the stable transfer function H(s)
#       Open red circles   : poles rejected because Re{p} > 0
#
# Only the N poles in the left half-plane are retained in H(s).
#
# For odd N:
#
#       one stable pole is real and negative and the remaining poles
#       occur in complex-conjugate pairs.
#
# For even N:
#
#       there is no real pole and all stable poles occur in
#       complex-conjugate pairs.
#
# This notebook intentionally displays only poles, not transmission zeros.
# ==============================================================================

# ==============================================================================
# DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML("""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:8px 10px;
    margin:0px 0px 8px 0px;
    font-size:13px;
    line-height:1.45;
    background-color:#f7fbff;
    width:1040px;
    max-width:1040px;
    box-sizing:border-box;
">
<b>Purpose:</b>
Visualize all 2N poles of H(s)H(-s) for a Chebyshev Type II low-pass filter and identify the N poles that form the stable transfer function H(s).
<br>
<b>Interpretation:</b>
Unlike Butterworth and Chebyshev Type I filters, the poles of a Chebyshev Type II filter do not generally lie on a simple conic such as a circle or ellipse. Their positions depend on N, ω<sub>s</sub> and A<sub>s</sub>. Filled red circles represent the poles in the left half-plane that are retained in H(s), while open red circles represent the corresponding right-half-plane poles that are rejected.
</div>
""", layout=Layout(width='1050px', max_width='1050px'))

# ==============================================================================
# CONTROLS
# ==============================================================================

slider_layout = Layout(width='280px')
style_opts = {'description_width':'95px'}

order_slider = IntSlider(min=2, max=10, step=1, value=5, description='Order N:', continuous_update=True, style=style_opts, layout=slider_layout)

ws_slider = FloatSlider(min=0.5, max=5.0, step=0.1, value=5.0, description='ωs:', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)

As_slider = FloatSlider(min=20.0, max=80.0, step=1.0, value=40.0, description='As (dB):', continuous_update=True, readout=True, readout_format='.0f', style=style_opts, layout=slider_layout)

parameter_title = HTML("""
<div style="
    font-size:14px;
    font-weight:bold;
    margin-top:3px;
    margin-bottom:5px;
">
Filter Parameters:
</div>
""")

info_html = HTML(layout=Layout(width='370px', max_width='370px'))

pole_table = HTML(layout=Layout(width='460px', max_width='460px'))

# ==============================================================================
# FIGURE
# ==============================================================================

fig, ax = plt.subplots(figsize=(6.6, 6.6))

used_scatter = ax.scatter([], [], s=95, marker='o', facecolors='red', edgecolors='red', linewidths=1.5, label='Used poles')

rejected_scatter = ax.scatter([], [], s=95, marker='o', facecolors='white', edgecolors='red', linewidths=1.8, label='Rejected poles')

ax.axhline(0.0, color='black', linewidth=0.9)
ax.axvline(0.0, color='black', linewidth=0.9)

ax.set_xlabel('Re{s}', fontsize=11)
ax.set_ylabel('Im{s}', fontsize=11)

ax.set_title('Chebyshev II Poles of H(s)H(-s)', fontsize=13, fontweight='bold', pad=8)

ax.grid(True, linestyle=':', alpha=0.35)

ax.set_aspect('equal', adjustable='box')

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10), ncol=2, fontsize=8)

axis_limit = 12.0

ax.set_xlim(-axis_limit, axis_limit)
ax.set_ylim(-axis_limit, axis_limit)

ax.set_xticks(np.arange(-12, 13, 3))
ax.set_yticks(np.arange(-12, 13, 3))

fig.subplots_adjust(left=0.12, right=0.96, bottom=0.17, top=0.92)

fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.resizable = False

fig.canvas.layout.width = '660px'
fig.canvas.layout.height = '660px'

# ==============================================================================
# UPDATE FUNCTION
# ==============================================================================

def update_chebyshev2_poles(change=None):

    N = order_slider.value
    ws = ws_slider.value
    As = As_slider.value

    # --------------------------------------------------------------------------
    # Ripple parameter
    # --------------------------------------------------------------------------

    epsilon = 1.0 / np.sqrt(10.0**(As / 10.0) - 1.0)

    # --------------------------------------------------------------------------
    # Auxiliary quantity
    # --------------------------------------------------------------------------

    mu = np.arcsinh(1.0 / epsilon) / N

    sinh_mu = np.sinh(mu)
    cosh_mu = np.cosh(mu)

    # --------------------------------------------------------------------------
    # Compute ALL 2N poles of H(s)H(-s)
    #
    # θk = (2k-1)π/(2N),  k = 1,...,2N
    #
    # The Chebyshev-II pole formulas follow from the inverse
    # transformation of the Chebyshev-I pole geometry.
    # --------------------------------------------------------------------------

    k = np.arange(1, 2 * N + 1)

    theta = (2 * k - 1) * np.pi / (2 * N)

    denominator = (sinh_mu**2) * (np.sin(theta)**2) + (cosh_mu**2) * (np.cos(theta)**2)

    sigma = -ws * sinh_mu * np.sin(theta) / denominator

    omega_poles = -ws * cosh_mu * np.cos(theta) / denominator

    left_poles = sigma + 1j * omega_poles

    # --------------------------------------------------------------------------
    # The formula above directly gives the LHP branch.
    #
    # Construct the corresponding RHP branch by origin symmetry:
    #
    #       p -> -p
    #
    # We keep the first N unique LHP poles and generate their symmetric
    # counterparts to obtain the full 2N pole set.
    # --------------------------------------------------------------------------

    lhp_candidates = left_poles[np.real(left_poles) < -1e-10]

    # Sort for consistent display
    lhp_candidates = lhp_candidates[np.argsort(np.angle(lhp_candidates))]

    # Remove possible numerical duplicates
    unique_lhp = []

    for p in lhp_candidates:

        if not any(np.abs(p - q) < 1e-8 for q in unique_lhp):
            unique_lhp.append(p)

    used_poles = np.array(unique_lhp[:N], dtype=complex)

    # If numerical selection does not return N because of ordering,
    # fall back to the direct stable branch from the first N angles.
    if len(used_poles) != N:

        theta_stable = (2 * np.arange(1, N + 1) - 1) * np.pi / (2 * N)

        denominator_stable = (sinh_mu**2) * (np.sin(theta_stable)**2) + (cosh_mu**2) * (np.cos(theta_stable)**2)

        sigma_stable = -ws * sinh_mu * np.sin(theta_stable) / denominator_stable

        omega_stable = -ws * cosh_mu * np.cos(theta_stable) / denominator_stable

        used_poles = sigma_stable + 1j * omega_stable

    rejected_poles = -used_poles

    poles = np.concatenate((used_poles, rejected_poles))

    # --------------------------------------------------------------------------
    # Update pole locations
    # --------------------------------------------------------------------------

    used_offsets = np.column_stack((np.real(used_poles), np.imag(used_poles)))

    rejected_offsets = np.column_stack((np.real(rejected_poles), np.imag(rejected_poles)))

    used_scatter.set_offsets(used_offsets)

    rejected_scatter.set_offsets(rejected_offsets)

    # --------------------------------------------------------------------------
    # Pole table
    # --------------------------------------------------------------------------

    rows = ""

    for index, p in enumerate(poles):

        if index < N:

            status = "USED"

            status_style = """
                color:#0066cc;
                background:#eef6ff;
                border:1px solid #9bc8f5;
            """

        else:

            status = "REJECTED"

            status_style = """
                color:#cc0000;
                background:#fff1f1;
                border:1px solid #efaaaa;
            """

        rows += f"""
        <tr style="border-bottom:1px solid #eeeeee;">

            <td style="
                padding:5px 8px;
                text-align:center;
                font-family:'Times New Roman',serif;
                font-size:17px;
                font-style:italic;
                white-space:nowrap;
            ">
                p<sub>{index}</sub>
            </td>

            <td style="
                padding:5px 10px;
                font-family:'Times New Roman',serif;
                font-size:16px;
                white-space:nowrap;
            ">
                {p.real:+.6f}
                <span style="font-style:italic;">{p.imag:+.6f}j</span>
            </td>

            <td style="
                padding:5px 8px;
                text-align:center;
            ">
                <span style="
                    {status_style}
                    padding:2px 7px;
                    border-radius:10px;
                    font-size:10px;
                    font-weight:bold;
                    letter-spacing:0.3px;
                    white-space:nowrap;
                ">
                    {status}
                </span>
            </td>

        </tr>
        """

    pole_table.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px;
        background:white;
        width:450px;
        max-height:620px;
        overflow-y:auto;
        box-sizing:border-box;
        font-size:12px;
    ">

    <div style="
        font-family:'Times New Roman',serif;
        font-size:18px;
        font-weight:bold;
        margin-bottom:7px;
        text-align:center;
    ">
        Pole Values
    </div>

    <table style="
        width:100%;
        border-collapse:collapse;
    ">

        <tr style="border-bottom:1px solid #bbbbbb;">
            <th style="padding:5px;">Pole</th>
            <th style="padding:5px;">Complex Value</th>
            <th style="padding:5px;">Status</th>
        </tr>

        {rows}

    </table>

    </div>
    """

    # --------------------------------------------------------------------------
    # Information panel
    # --------------------------------------------------------------------------

    if N % 2 == 1:

        parity_text = "Odd order"

        real_candidates = used_poles[np.abs(np.imag(used_poles)) < 1e-8]

        if len(real_candidates) > 0:
            real_pole_text = f"One stable real pole at s = {real_candidates[0].real:.4f}"
        else:
            real_pole_text = "One stable real pole"

    else:

        parity_text = "Even order"
        real_pole_text = "No real pole"

    # Corresponding Chebyshev-I ellipse parameters for comparison
    alpha_I = ws * np.sinh(mu)
    beta_I = ws * np.cosh(mu)

    info_html.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px 10px;
        margin-top:8px;
        font-size:12px;
        line-height:1.65;
        background:white;
        width:365px;
        box-sizing:border-box;
    ">

    <div>
        <b>Filter:</b>
        <span style="color:#0066cc;">Chebyshev Type II low-pass</span>
    </div>

    <div>
        <b>Order N:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Stopband edge:</b>
        <span style="color:#0066cc;">ωs = {ws:.2f} rad/s</span>
    </div>

    <div>
        <b>Stopband attenuation:</b>
        <span style="color:#0066cc;">As = {As:.1f} dB</span>
    </div>

    <div>
        <b>Ripple parameter:</b>
        <span style="color:#0066cc;">ε = {epsilon:.8f}</span>
    </div>

    <div>
        <b>Auxiliary μ:</b>
        <span style="color:#0066cc;">{mu:.6f}</span>
    </div>

    <div style="
        margin-top:5px;
        padding-top:5px;
        border-top:1px solid #eeeeee;
    ">
        <b>Pole geometry:</b>
    </div>

    <div>
        <span style="color:#0066cc;">
        The poles do not generally lie on a circle, ellipse or hyperbola.
        </span>
    </div>

    <div>
        <b>Associated Chebyshev-I α:</b>
        <span style="color:#0066cc;">{alpha_I:.6f}</span>
    </div>

    <div>
        <b>Associated Chebyshev-I β:</b>
        <span style="color:#0066cc;">{beta_I:.6f}</span>
    </div>

    <div style="
        margin-top:5px;
        padding-top:5px;
        border-top:1px solid #eeeeee;
    ">
        <b>Total poles:</b>
        <span style="color:#0066cc;">{2 * N}</span>
    </div>

    <div>
        <b>Used poles:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Rejected poles:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Order type:</b>
        <span style="color:#0066cc;">{parity_text}</span>
    </div>

    <div>
        <b>Real pole:</b>
        <span style="color:#0066cc;">{real_pole_text}</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Observation:</b><br>
        The Chebyshev-II pole pattern results from an inverse-type
        transformation of the Chebyshev-I geometry and therefore is generally
        not a conic. Changing N or As changes the geometry, whereas changing
        ωs mainly scales the pattern. Only the N left-half-plane poles are
        retained in the stable transfer function H(s).
    </div>

    </div>
    """

    # --------------------------------------------------------------------------
    # Redraw
    # --------------------------------------------------------------------------

    fig.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

order_slider.observe(update_chebyshev2_poles, names='value')
ws_slider.observe(update_chebyshev2_poles, names='value')
As_slider.observe(update_chebyshev2_poles, names='value')

# ==============================================================================
# LAYOUT
# ==============================================================================

controls = VBox([parameter_title, order_slider, ws_slider, As_slider, info_html], layout=Layout(width='380px', min_width='380px', max_width='380px', flex='0 0 380px', align_items='flex-start'))

main_row = HBox([controls, fig.canvas, pole_table], layout=Layout(width='1510px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# INITIALIZE
# ==============================================================================

update_chebyshev2_poles()

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)
display(main_row)